# Tổng hợp kết quả lựa chọn đặc trưng

Notebook này chỉ đọc các artifact đã có trong `outputs_baseline_full/` và `outputs_baseline_split/`, không chạy lại mRMR/Boruta và không thay đổi pipeline.

- K-Fold: `k = 2, 4, 5, 6`, gồm Raw, mRMR-50, mRMR-75 và Boruta.
- Train-test: `k = 4, 5`, gồm mRMR-50, mRMR-75 và Boruta; Raw không được xuất trong flow này.
- Boruta xuất riêng số Confirmed, Tentative, Rejected và tập cuối theo `resolved_selection_mode`.
- Các file CSV được ghi vào `outputs_LV/feature_selection_summary/`.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

def find_repo_root():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for p in candidates:
        if (p / 'outputs_baseline_full').is_dir() and (p / 'outputs_baseline_split').is_dir():
            return p
    raise FileNotFoundError('Không tìm thấy repository root chứa outputs_baseline_full và outputs_baseline_split.')

REPO_ROOT = find_repo_root()
FULL_ROOT = REPO_ROOT / 'outputs_baseline_full'
SPLIT_ROOT = REPO_ROOT / 'outputs_baseline_split'
EXPORT_ROOT = REPO_ROOT / 'outputs_LV' / 'feature_selection_summary'
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
KFOLD_K = [2, 4, 5, 6]
SPLIT_K = [4, 5]
METHODS_KFOLD = ['raw', 'mrmr_k50', 'mrmr_k75', 'boruta']
METHODS_SPLIT = ['mrmr_k50', 'mrmr_k75', 'boruta']
print(f'REPO_ROOT = {REPO_ROOT}')
print(f'EXPORT_ROOT = {EXPORT_ROOT}')

In [ ]:
def load_json(path):
    if not path.exists():
        return {}
    with path.open(encoding='utf-8') as f:
        return json.load(f)

def ordered_datasets(names):
    # Đặt các bộ CuMiDa trước, sau đó đến GEO; trong mỗi nhóm sắp xếp theo tên.
    return sorted(names, key=lambda x: (0 if x.lower().startswith('cumida-') else 1, x.lower()))

def method_dir(flow_root, k, dataset, method):
    return flow_root / f'k{k}' / 'feature_selection' / dataset / method

def read_feature_list(method_path, method):
    selected_path = method_path / 'selected_features' / 'selected_features.json'
    data = load_json(selected_path)
    if method == 'raw':
        metadata = load_json(method_path / 'params' / 'metadata.json')
        n = int(metadata.get('n_selected_features', metadata.get('n_original_features', 0)) or 0)
        ranking = method_path / 'selected_features' / 'ranking.csv'
        if ranking.exists():
            return pd.read_csv(ranking)['Feature'].astype(str).tolist()
        return [f'__RAW_FEATURE_COUNT__:{n}'] if n else []
    return [str(x) for x in data.get('selected_features', [])]

def read_boruta(method_path):
    metadata = load_json(method_path / 'params' / 'metadata.json')
    selected = load_json(method_path / 'selected_features' / 'selected_features.json')
    confirmed = [str(x) for x in selected.get('confirmed', [])]
    tentative = [str(x) for x in selected.get('tentative', [])]
    rejected = int(metadata.get('rejected', 0) or 0)
    # Fallback cho các artifact cũ thiếu một số metadata.
    if not confirmed and (method_path / 'selected_features' / 'support.csv').exists():
        support = pd.read_csv(method_path / 'selected_features' / 'support.csv')
        confirmed = support.loc[support['Support'].astype(bool), 'Feature'].astype(str).tolist()
        tentative = support.loc[support['Support_Weak'].astype(bool), 'Feature'].astype(str).tolist()
        rejected = int(len(support) - len(confirmed) - len(tentative))
    final_features = [str(x) for x in selected.get('selected_features', [])]
    if not final_features:
        mode = metadata.get('resolved_selection_mode', metadata.get('selection_mode', 'confirmed'))
        final_features = confirmed + tentative if mode == 'confirmed_tentative' else confirmed
    return {
        'confirmed': confirmed,
        'tentative': tentative,
        'rejected': rejected,
        'confirmed_tentative': confirmed + tentative,
        'final': final_features,
        'selection_mode': metadata.get('selection_mode', selected.get('selection_mode', '')),
        'resolved_selection_mode': metadata.get('resolved_selection_mode', selected.get('resolved_selection_mode', '')),
        'n_samples': metadata.get('n_samples'),
        'n_original_features': metadata.get('n_original_features'),
    }

In [ ]:
def collect_flow(flow_name, root, k_values, methods):
    summary_rows, feature_rows = [], []
    dataset_names = set()
    for k in k_values:
        fs_root = root / f'k{k}' / 'feature_selection'
        if fs_root.exists():
            dataset_names.update(p.name for p in fs_root.iterdir() if p.is_dir())
    for dataset in ordered_datasets(dataset_names):
        for k in k_values:
            row = {'flow': flow_name, 'dataset': dataset, 'k': k}
            for method in methods:
                p = method_dir(root, k, dataset, method)
                if not p.exists():
                    continue
                metadata = load_json(p / 'params' / 'metadata.json')
                row.setdefault('n_samples', metadata.get('n_samples'))
                row.setdefault('n_original_features', metadata.get('n_original_features'))
                if method == 'boruta':
                    b = read_boruta(p)
                    row.update({
                        'boruta_confirmed': len(b['confirmed']),
                        'boruta_tentative': len(b['tentative']),
                        'boruta_rejected': b['rejected'],
                        'boruta_confirmed_tentative': len(b['confirmed_tentative']),
                        'boruta_final': len(b['final']),
                        'boruta_selection_mode': b['selection_mode'],
                        'boruta_resolved_selection_mode': b['resolved_selection_mode'],
                    })
                    groups = [('Confirmed', b['confirmed']), ('Tentative', b['tentative']), ('Final', b['final'])]
                else:
                    features = read_feature_list(p, method)
                    row[f'{method}_selected'] = len(features)
                    groups = [] if method == 'raw' else [('Selected', features)]
                for decision, features in groups:
                    for rank, feature in enumerate(features, start=1):
                        feature_rows.append({
                            'flow': flow_name, 'dataset': dataset, 'k': k,
                            'method': method, 'decision': decision,
                            'feature': feature, 'rank_in_group': rank,
                            'source_path': str(p)
                        })
            if len(row) > 3:
                summary_rows.append(row)
    summary = pd.DataFrame(summary_rows)
    features = pd.DataFrame(feature_rows)
    return summary, features

kfold_summary, kfold_features = collect_flow('kfold', FULL_ROOT, KFOLD_K, METHODS_KFOLD)
split_summary, split_features = collect_flow('train_test', SPLIT_ROOT, SPLIT_K, METHODS_SPLIT)
print('K-Fold:', kfold_summary.shape, kfold_features.shape)
print('Train-test:', split_summary.shape, split_features.shape)

In [ ]:
def method_feature_map(features_df, flow, dataset, k, method, decision=None):
    q = features_df[(features_df.flow == flow) & (features_df.dataset == dataset) & (features_df.k == k) & (features_df.method == method)]
    if decision is not None:
        q = q[q.decision == decision]
    return set(q.feature.astype(str))

def jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else None

def make_overlap(summary, features, flow):
    rows = []
    for _, r in summary.iterrows():
        ds, k = r['dataset'], int(r['k'])
        sets = {
            'mrmr_k50': method_feature_map(features, flow, ds, k, 'mrmr_k50'),
            'mrmr_k75': method_feature_map(features, flow, ds, k, 'mrmr_k75'),
            'boruta_final': method_feature_map(features, flow, ds, k, 'boruta', 'Final'),
        }
        for a, b in [('mrmr_k50', 'mrmr_k75'), ('mrmr_k50', 'boruta_final'), ('mrmr_k75', 'boruta_final')]:
            rows.append({'flow': flow, 'dataset': ds, 'k': k, 'set_a': a, 'set_b': b,
                         'intersection_count': len(sets[a] & sets[b]), 'jaccard': jaccard(sets[a], sets[b])})
    return pd.DataFrame(rows)

def make_full_vs_train(full_summary, full_features, split_summary, split_features):
    rows = []
    common = sorted(set(full_summary.dataset) & set(split_summary.dataset), key=lambda x: (0 if x.lower().startswith('cumida-') else 1, x.lower()))
    for ds in common:
        for k in [4, 5]:
            for method in ['mrmr_k50', 'mrmr_k75', 'boruta']:
                decision = 'Final' if method == 'boruta' else 'Selected'
                a = method_feature_map(full_features, 'kfold', ds, k, method, decision)
                b = method_feature_map(split_features, 'train_test', ds, k, method, decision)
                rows.append({'dataset': ds, 'k': k, 'method': method, 'full_count': len(a),
                             'train_only_count': len(b), 'intersection_count': len(a & b),
                             'jaccard': jaccard(a, b)})
    return pd.DataFrame(rows)

kfold_overlap = make_overlap(kfold_summary, kfold_features, 'kfold')
split_overlap = make_overlap(split_summary, split_features, 'train_test')
full_vs_train = make_full_vs_train(kfold_summary, kfold_features, split_summary, split_features)

In [ ]:
# Chuẩn hóa thứ tự cột để dùng trực tiếp cho bảng luận văn.
summary_columns = [
    'flow', 'dataset', 'k', 'n_samples', 'n_original_features',
    'raw_selected', 'mrmr_k50_selected', 'mrmr_k75_selected',
    'boruta_confirmed', 'boruta_tentative', 'boruta_rejected',
    'boruta_confirmed_tentative', 'boruta_final',
    'boruta_selection_mode', 'boruta_resolved_selection_mode'
]
def finalize_summary(df):
    for c in summary_columns:
        if c not in df.columns:
            df[c] = pd.NA
    return df[summary_columns].sort_values(['flow', 'dataset', 'k'], key=lambda s: s.map(lambda x: (0, str(x).lower()) if str(x).lower().startswith('cumida-') else (1, str(x).lower())) if s.name == 'dataset' else s).reset_index(drop=True)

kfold_summary = finalize_summary(kfold_summary)
split_summary = finalize_summary(split_summary)
kfold_summary.to_csv(EXPORT_ROOT / 'kfold_feature_selection_summary.csv', index=False, encoding='utf-8-sig')
split_summary.to_csv(EXPORT_ROOT / 'train_test_feature_selection_summary.csv', index=False, encoding='utf-8-sig')
kfold_features.to_csv(EXPORT_ROOT / 'kfold_feature_selection_features.csv', index=False, encoding='utf-8-sig')
split_features.to_csv(EXPORT_ROOT / 'train_test_feature_selection_features.csv', index=False, encoding='utf-8-sig')
kfold_overlap.to_csv(EXPORT_ROOT / 'kfold_feature_selection_overlap.csv', index=False, encoding='utf-8-sig')
split_overlap.to_csv(EXPORT_ROOT / 'train_test_feature_selection_overlap.csv', index=False, encoding='utf-8-sig')
full_vs_train.to_csv(EXPORT_ROOT / 'full_vs_train_feature_selection_overlap.csv', index=False, encoding='utf-8-sig')

print('Đã xuất:')
for p in sorted(EXPORT_ROOT.glob('*.csv')):
    print(f'  {p.name}: {pd.read_csv(p).shape}')

In [ ]:
display(kfold_summary.head(12))
display(split_summary.head(12))
display(kfold_overlap.head())
display(full_vs_train.head())